# 00. Start the services

Starts MinIO and the Iceberg REST catalog with Docker Compose and waits until both answer.

## What to look at

- The **health table**: all three services should say `up`.
- The **URL table**: these are the pages to keep open in the browser while you demo.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
r = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True)
print(r.stdout[-600:] + r.stderr[-1200:])

In [ ]:
# the project's own wait script (polls MinIO + the REST catalog)
r = subprocess.run(["scripts/wait_for_services.sh"], capture_output=True, text=True)
print(r.stdout + r.stderr)
assert r.returncode == 0, "services did not become ready"

In [ ]:
import urllib.request
def status(url):
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            return "up"
    except urllib.error.HTTPError:
        return "up"          # it answered (e.g. a redirect or 4xx on a bare path)
    except Exception:
        return "not running"
pd.DataFrame([
    ("MinIO S3 API",          common.S3_ENDPOINT + "/minio/health/live", status(common.S3_ENDPOINT + "/minio/health/live")),
    ("MinIO console",         MINIO_CONSOLE,                             status(MINIO_CONSOLE)),
    ("Iceberg REST catalog",  common.CATALOG_URI + "/v1/config",         status(common.CATALOG_URI + "/v1/config")),
], columns=["service", "url", "health"])

In [ ]:
pd.DataFrame([
    ("MinIO console (files in the lake)", "http://localhost:9001", "login minioadmin / minioadmin"),
    ("Spark UI (jobs, scan metrics)",     "http://localhost:4040", "exists only while a Spark session is running (any notebook after 01)"),
    ("Search UI (visual search demo)",    "http://localhost:8765", "starts with `make ui`, or run the optional cell below"),
], columns=["page", "url", "note"])

In [ ]:
# OPTIONAL: start the search UI in the background (needs the gold table, i.e. notebooks 02-07 done).
START_SEARCH_UI = False
if START_SEARCH_UI:
    import tempfile
    log = open(Path(tempfile.gettempdir()) / "search_ui.log", "w")
    subprocess.Popen([sys.executable, "src/ui_server.py"], stdout=log, stderr=log, cwd=ROOT)
    print("starting... open http://localhost:8765 in ~30 seconds")

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Log in. You should see one bucket, **warehouse**, with nothing in it yet (or the tables from an earlier run). The bucket was created by the one-shot `mc` container in docker-compose.